In [1]:
from datetime import datetime, timedelta, date
import requests
import pandas as pd
import pickle
import numpy as np
import holidays

csv_te_voorspellen = './data csv/TEST_kijkcijfers.csv'

In [2]:
def ophalenKijkcijferData(startDate, endDate):
  print(f"Ophalen kijkcijfer-data van {startDate.year}-{startDate.month}-{startDate.day} tot {endDate.year}-{endDate.month}-{endDate.day}")
  kijkcijfersData = []
  #elke dag ophalen (startDate is huidige dag)
  while startDate <= endDate:
    datum = f"{startDate.year}-{startDate.month}-{startDate.day}"
    url = f"https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={datum}&reportType=north"

    try:
      response = requests.get(url)
      if response.status_code == 200:
        data = response.json()
        programmaLijst = data.get('hydra:member', [])
                        
        for programma in programmaLijst:
          try:
            kijkcijfersData.append({
              'dateDiff': programma.get('dateDiff'),
              'ranking': programma.get('ranking'),
              'description': programma.get('description'),
              'channel': programma.get('channel'),
              'startTime': programma.get('startTime'),
              'rLength': programma.get('rLength'),
              'rateInK': programma.get('rateInK'),
              'live': programma.get('live')
            })
                                   
          except Exception as e:
            print(f"error {datum}: {e}")         
      else:
        print(f"no data {datum}")
                        
    except Exception as e:
      print(f"error: {e}")
    
    startDate += timedelta(days=1)

  print("KijkcijferData opgehaald")
  df = pd.DataFrame(kijkcijfersData)

  return df

def ophalenWeerData(startDate, endDate):
    latitude = 51.05
    longitude = 3.7167
    today = datetime.today().date()

    hourly_vars = [
        "temperature_2m", "apparent_temperature", "weather_code", "precipitation",
        "rain", "snowfall", "cloud_cover", "windspeed_10m", "sunshine_duration"
    ]
    
    common_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": hourly_vars,
        "timezone": "Europe/Brussels",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "windspeed_unit": "kmh"
    }

    def fetch_weather_data(api_url, start, end):
        params = common_params.copy()
        params.update({
            "start_date": start.strftime('%Y-%m-%d'),
            "end_date": end.strftime('%Y-%m-%d')
        })
        response = requests.get(api_url, params=params)
        if response.status_code == 200:
            data = response.json().get("hourly", {})
            df = pd.DataFrame({var: data.get(var, []) for var in hourly_vars})
            df["timestamp"] = pd.to_datetime(data.get("time", []))
            if not df.empty:
                df["hour"] = df["timestamp"].dt.hour
                df["day_of_week"] = df["timestamp"].dt.dayofweek
                df["month"] = df["timestamp"].dt.month
                df["year"] = df["timestamp"].dt.year
            return df
        else:
            print(f"Fout bij ophalen data: {response.status_code}")
            print(response.text)
            return pd.DataFrame()

    dataframes = []

    # Historische data
    if startDate.date() < today:
        print("Ophalen historische data")
        hist_end = min(endDate.date(), today - timedelta(days=1))
        dataframes.append(fetch_weather_data(
            "https://archive-api.open-meteo.com/v1/archive",
            startDate, datetime.combine(hist_end, datetime.min.time())
        ))

    # Forecast data
    if endDate.date() >= today:
        print("Ophalen forecast data")
        forecast_start = max(endDate, datetime.combine(today, datetime.min.time()))
        print(forecast_start)
        dataframes.append(fetch_weather_data(
            "https://api.open-meteo.com/v1/forecast",
            forecast_start, endDate
        ))

    if dataframes:
        print("Weerdata opgehaald")
        return pd.concat(dataframes).sort_values("timestamp").reset_index(drop=True)
    else:
        return pd.DataFrame()


In [3]:
def testCSV(csv):
    df = pd.read_csv(csv, delimiter=';')
    # Kolommen hernoemen
    df = df.rename(columns={
        'Programma': 'description',
        'Zender': 'channel',
        'Datum ': 'dateDiff',
        'Start ': 'startTime',
        'Duur': 'rLength'
    })
    # Datum en tijd samenvoegen tot datetime
    df['dateDiff'] = pd.to_datetime(df['dateDiff'], dayfirst=True)
    # Starttijd naar HH:MM:SS
    df['startTime'] = df['startTime'].str[:8]
    # Duur naar HH:MM:SS
    df['rLength'] = df['rLength'].apply(lambda x: str(pd.to_timedelta(x)))
    # Voeg dummy kolommen toe als nodig
    df['ranking'] = 0
    df['live'] = 0
    df['Kijkers'] = None

    return df[['dateDiff', 'ranking', 'description', 'channel', 'startTime', 'rLength', 'live']]


In [4]:
teVoorspellen = testCSV(csv_te_voorspellen)

teVoorspellen['dateDiff'] = pd.to_datetime(teVoorspellen['dateDiff'])

end_date = teVoorspellen['dateDiff'].max()
start_date = end_date - timedelta(weeks=3)
histKijkcijfers = ophalenKijkcijferData(start_date, end_date - timedelta(days=1))
histWeerdata = ophalenWeerData(start_date, end_date)

Ophalen kijkcijfer-data van 2025-4-15 tot 2025-5-5
KijkcijferData opgehaald
Ophalen historische data
Weerdata opgehaald


In [5]:
def cleanKijkcijferData(df):

    # Zet 'Kijkers' kolom, als 'rateInK' bestaat
    if 'rateInK' in df.columns:
        df['Kijkers'] = (
            df['rateInK']
            .dropna()
            .astype(str)
            .str.replace('.', '', regex=False)
            .astype(int)
        )
    else:
        df['Kijkers'] = None

    # rLength aanpassen
    df['rLength'] = df['rLength'].astype(str).apply(
    lambda x: x[-8:] if 'days' in x else x.zfill(8)
    )

    # Tijd aanpassen
    tijd_regex = r'^\d{2}:\d{2}:\d{2}$'
    # Omzetten naar datetime
    df['date'] = pd.to_datetime(df['dateDiff']).dt.date
    # Filter rijen met formaat
    df = df[df['startTime'].str.match(tijd_regex, na=False) & df['rLength'].str.match(tijd_regex, na=False)].copy()
    
    # Afleveringlengte naar seconden omzetten 
    df['Lengte_sec'] = pd.to_timedelta(df['rLength']).dt.total_seconds().astype(int)

    # Uren met 24+
    def time_cor(rij):
        tijdArr = rij['startTime'].split(':')
        if int(tijdArr[0]) >= 24:
            tijdArr[0] = str(int(tijdArr[0]) - 24).zfill(2)
            rij['date'] += timedelta(days=1)
        rij['startTime'] = ':'.join(tijdArr)
        return rij
    
    df = df.apply(time_cor, axis=1)

    # 1 kolom voor beide data
    df['FullDate'] = pd.to_datetime(df['date'].astype(str) 
                                    + " " + df['startTime'].astype(str))
    
    # Hour en minute voor join later on
    df['hour'] = pd.to_datetime(df['startTime'], format='%H:%M:%S').dt.hour
    df['minute'] = 0

    # Kolommen verwijderen die niet nodig meer zijn, als ze bestaan
    columns_to_drop = ['startTime', 'rLength', 'rateInK', 'ranking', 'live']
    df.drop([col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)


    # De nieuwe dataframe
    df = df[['FullDate', 'date', 'hour', 'minute', 'channel', 'description', 'Lengte_sec', 'Kijkers']]

    # Hernoemen kolommen
    df.rename(columns={'description': 'Programma', 'channel': 'Kanaal'}, inplace=True)

    return df


def cleanWeerData(df):
  weerData = df
  weerData['timestamp'] = pd.to_datetime(weerData['timestamp'])
  #naar zelfde formaat als kijkcijfer datum
  weerData['datetime'] = weerData['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

  #hour voor join later on
  weerData['hour'] = pd.to_datetime(weerData['datetime']).dt.hour
  weerData['minute'] = pd.to_datetime(weerData['datetime']).dt.minute
  weerData['date'] = pd.to_datetime(weerData['datetime']).dt.date

  #verwijder kolom
  weerData = weerData.drop(columns=['timestamp'])

  weerData = weerData[['datetime', 'date' ,'hour', 'minute', 'temperature_2m', 'apparent_temperature', 
                            'rain', 'snowfall', 'weather_code', 'cloud_cover', 
                            'windspeed_10m', 'sunshine_duration']]

  #hernoemen kolommen
  weerData.rename(columns={'temperature_2m':'Temperatuur', 'apparent_temperature':'Gevoelstemp', 'windspeed_10m': 'Windsnelheid', 'rain':'Regen', 'snowfall': 'Sneeuw', 'weather_code':'Weercode', 'cloud_cover':'Bewolking', 'sunshine_duration':'Zonnenschijn'}, inplace=True)

  return weerData

In [56]:
histWeerdataClean = cleanWeerData(histWeerdata)
histKijkcijfersClean = cleanKijkcijferData(histKijkcijfers)
teVoorspellenClean = cleanKijkcijferData(teVoorspellen)
print(histWeerdataClean)
print(histKijkcijfersClean)
print(teVoorspellenClean)

                datetime        date  hour  minute  Temperatuur  Gevoelstemp  \
0    2025-04-15 00:00:00  2025-04-15     0       0         12.3         10.0   
1    2025-04-15 01:00:00  2025-04-15     1       0         11.6          9.4   
2    2025-04-15 02:00:00  2025-04-15     2       0         11.1          8.8   
3    2025-04-15 03:00:00  2025-04-15     3       0         10.8          8.1   
4    2025-04-15 04:00:00  2025-04-15     4       0         10.7          8.1   
5    2025-04-15 05:00:00  2025-04-15     5       0         10.9          8.3   
6    2025-04-15 06:00:00  2025-04-15     6       0         11.2          8.5   
7    2025-04-15 07:00:00  2025-04-15     7       0         11.0          8.5   
8    2025-04-15 08:00:00  2025-04-15     8       0         11.3          9.4   
9    2025-04-15 09:00:00  2025-04-15     9       0         13.0         11.4   
10   2025-04-15 10:00:00  2025-04-15    10       0         15.1         13.6   
11   2025-04-15 11:00:00  2025-04-15    

In [57]:
def mergen(kijkcijfers, weer):
  kijkcijfersWeer = pd.merge(kijkcijfers, weer, on=['date', 'hour'], how='left')
  kijkcijfersWeer = kijkcijfersWeer[['FullDate', 'date', 'hour', 'Kanaal', 'Programma', 'Lengte_sec', 'Kijkers', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Sneeuw', 'Weercode', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  kijkcijfersWeer.dropna(inplace=True)
  return kijkcijfersWeer

In [58]:
histKijkcijfersWeer = mergen(histKijkcijfersClean, histWeerdataClean)
print(histKijkcijfersWeer)

               FullDate        date  hour                  Kanaal  \
0   2025-04-15 20:14:58  2025-04-15    20                   VRT 1   
1   2025-04-15 20:41:25  2025-04-15    20                   VRT 1   
2   2025-04-15 19:00:05  2025-04-15    19                   VRT 1   
3   2025-04-15 21:27:50  2025-04-15    21                   VRT 1   
4   2025-04-15 20:44:56  2025-04-15    20                     VTM   
5   2025-04-15 19:45:34  2025-04-15    19                   VRT 1   
6   2025-04-15 20:09:04  2025-04-15    20                     VTM   
7   2025-04-15 18:59:48  2025-04-15    18                     VTM   
8   2025-04-15 18:28:21  2025-04-15    18                   VRT 1   
9   2025-04-15 21:19:26  2025-04-15    21                   PLAY4   
10  2025-04-15 13:00:04  2025-04-15    13                   VRT 1   
11  2025-04-15 18:24:52  2025-04-15    18                     VTM   
12  2025-04-15 20:00:03  2025-04-15    20              VRT CANVAS   
13  2025-04-15 22:02:15  2025-04-1

In [59]:
teVoorspellenData = pd.merge(teVoorspellenClean, histWeerdataClean, on=['date', 'hour', 'minute'], how='left')
teVoorspellenData = teVoorspellenData.drop(columns=['datetime', 'minute'])
print(teVoorspellenData)

             FullDate        date  hour Kanaal            Programma  \
0 2025-05-06 20:17:52  2025-05-06    20  VRT 1                THUIS   
1 2025-05-06 19:00:04  2025-05-06    19  VRT 1   HET 7 UUR-JOURNAAL   
2 2025-05-06 19:46:05  2025-05-06    19  VRT 1        MAN BIJT HOND   
3 2025-05-06 18:29:07  2025-05-06    18  VRT 1              BLOKKEN   
4 2025-05-06 18:59:48  2025-05-06    18    VTM       NIEUWS 19U VTM   
5 2025-05-06 21:37:08  2025-05-06    21  VRT 1   DE DAG VAN VANDAAG   
6 2025-05-06 20:05:28  2025-05-06    20    VTM              FAMILIE   
7 2025-05-06 20:41:26  2025-05-06    20    VTM         HUIS GEMAAKT   
8 2025-05-06 20:44:02  2025-05-06    20  VRT 1  GELUKKIG GESCHEIDEN   
9 2025-05-06 13:00:04  2025-05-06    13  VRT 1   HET 1 UUR-JOURNAAL   

   Lengte_sec Kijkers  Temperatuur  Gevoelstemp  Regen  Sneeuw  Weercode  \
0        1470    None         12.6          9.1    0.0     0.0         3   
1        2598    None         13.0          9.3    0.0     0.0    

In [61]:
def tijdFeatures(df):
    df['date'] = pd.to_datetime(df['date'])
    #feestdagen
    feestdagen = holidays.BE()
    df['isFeestdag'] = df['date'].apply(lambda x: 1 if x in feestdagen else 0)
    #dag van de week
    df['Weekdag'] = df['date'].dt.weekday
    #weekend
    df['isWeekend'] = df['Weekdag'].apply(lambda x: 1 if x >= 5 else 0)
    #seizoenen
    df['Seizoen'] = df['date'].apply(seizoenFinder)

    return df

#seizoen
def seizoenFinder(datum):
    inputDatum = datum.date()
    Y = inputDatum.year
    seizoenen = {
        'lente': (date(Y, 3, 20), date(Y, 6, 20)),
        'zomer': (date(Y, 6, 21), date(Y, 9, 22)),
        'herfst':   (date(Y, 9, 23), date(Y, 12, 20)),
        'winter': (date(Y, 12, 21), date(Y + 1, 3, 19)),
    }

    for seizoen, (start, end) in seizoenen.items():
        if start <= inputDatum <= end:
            return seizoen
    return 'winter'

In [63]:
teVoorspellenData = tijdFeatures(teVoorspellenData)
histKijkcijfersWeer = tijdFeatures(histKijkcijfersWeer)
histWeerdataClean = tijdFeatures(histWeerdataClean)
histWeerdataClean.drop(columns=['minute'], inplace=True)
display(teVoorspellenData.head())
display(histKijkcijfersWeer.head())
display(histWeerdataClean.head())

,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
0,2025-05-06 20:17:52,2025-05-06,20,VRT 1,THUIS,1470,None,12.6,9.1,0.0,0.0,3,100,18.0,0.0,0,1,0,lente
1,2025-05-06 19:00:04,2025-05-06,19,VRT 1,HET 7 UUR-JOURNAAL,2598,None,13.0,9.3,0.0,0.0,3,100,18.8,0.0,0,1,0,lente
2,2025-05-06 19:46:05,2025-05-06,19,VRT 1,MAN BIJT HOND,1410,None,13.0,9.3,0.0,0.0,3,100,18.8,0.0,0,1,0,lente
3,2025-05-06 18:29:07,2025-05-06,18,VRT 1,BLOKKEN,1710,None,13.8,10.0,0.1,0.0,51,100,17.3,0.0,0,1,0,lente
4,2025-05-06 18:59:48,2025-05-06,18,VTM,NIEUWS 19U VTM,3201,None,13.8,10.0,0.1,0.0,51,100,17.3,0.0,0,1,0,lente


,FullDate,date,hour,Kanaal,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
0,2025-04-15 20:14:58,2025-04-15,20,VRT 1,THUIS,1469,988011,16.6,14.5,0.0,0.0,3,98,13.6,3600.0,0,1,0,lente
1,2025-04-15 20:41:25,2025-04-15,20,VRT 1,FACTCHECKERS,2654,889262,16.6,14.5,0.0,0.0,3,98,13.6,3600.0,0,1,0,lente
2,2025-04-15 19:00:05,2025-04-15,19,VRT 1,HET 7 UUR-JOURNAAL,2501,810136,18.0,16.2,0.0,0.0,3,90,11.1,3600.0,0,1,0,lente
3,2025-04-15 21:27:50,2025-04-15,21,VRT 1,KNOKKE OFF,1895,750554,14.9,13.0,0.0,0.0,2,63,12.3,0.0,0,1,0,lente
4,2025-04-15 20:44:56,2025-04-15,20,VTM,HUIS GEMAAKT,3346,609911,16.6,14.5,0.0,0.0,3,98,13.6,3600.0,0,1,0,lente


,datetime,date,hour,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isFeestdag,Weekdag,isWeekend,Seizoen
0,2025-04-15 00:00:00,2025-04-15,0,12.3,10.0,0.0,0.0,3,81,9.8,0.0,0,1,0,lente
1,2025-04-15 01:00:00,2025-04-15,1,11.6,9.4,0.0,0.0,1,44,9.5,0.0,0,1,0,lente
2,2025-04-15 02:00:00,2025-04-15,2,11.1,8.8,0.0,0.0,1,32,10.1,0.0,0,1,0,lente
3,2025-04-15 03:00:00,2025-04-15,3,10.8,8.1,0.0,0.0,3,90,11.6,0.0,0,1,0,lente
4,2025-04-15 04:00:00,2025-04-15,4,10.7,8.1,0.0,0.0,3,98,11.8,0.0,0,1,0,lente


In [121]:
def lagFeatures(df, hist):
  df['Kijkers'] = None
  df = pd.concat([hist, df], ignore_index=True)
  # Sorteer op tijd binnen elke groep
  df = df.sort_values(['Programma', 'FullDate'])

  df = df.sort_values(['Programma', 'FullDate'])

  # Bereken gemiddelde kijkers per Programma (op basis van historische data)
  kijkers_mean = df.groupby('Programma')['Kijkers'].transform('mean')

  # Bereken lag features per programma
  for i in range(1, 4):
      df[f'Kijkers_lag_{i}'] = df.groupby('Programma')['Kijkers'].shift(i)
      df[f'Kijkers_lag_{i}'] = df[f'Kijkers_lag_{i}'].fillna(kijkers_mean).round().astype(int)

  return df

In [122]:
pred_hist_df = lagFeatures(teVoorspellenData, histKijkcijfersWeer)
print("lagfeatures toegevoegd: ")
display(pred_hist_df.dtypes)
display(pred_hist_df.sort_values(by='FullDate').tail(10))
display(pred_hist_df[pred_hist_df.isna().any(axis=1)])



C:\Users\krist\AppData\Local\Temp\ipykernel_10392\1906661763.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[f'Kijkers_lag_{i}'] = df[f'Kijkers_lag_{i}'].fillna(kijkers_mean).round().astype(int)


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [14]:
def oneHot(df):
  with open('./models/oneHotEncoder.pkl', 'rb') as oneHotFile:
    oneHotEnc = pickle.load(oneHotFile)

  lageKard = df[[ 'hour','Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen']]
  dfOneHot = oneHotEnc.transform(lageKard)

  oneHotOutp = pd.DataFrame(dfOneHot.toarray(), 
                            columns=oneHotEnc.get_feature_names_out(), 
                            index=lageKard.index)

  df = df.drop(columns=['hour', 'Kanaal', 'isFeestdag', 'Weekdag', 'Seizoen'])
  df = pd.concat([df, oneHotOutp], axis = 1)
  return df

def target(df):
  #target encoding voor medium kardinaliteiten
  with open('./models/oneHotTarget.pkl', 'rb') as f:
    targetEnc = pickle.load(f)
  medKardinaliteit = df[['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn']]
  #verdere feature engineering op vorig model
  target = targetEnc.transform(medKardinaliteit)
  df = df.drop(columns=['date', 'Programma', 'Lengte_sec', 'Temperatuur', 'Gevoelstemp', 'Regen', 'Bewolking', 'Windsnelheid', 'Zonnenschijn'])
  f = pd.concat([df, target], axis=1)

  return f

In [35]:
teVoorspellenData = oneHot(pred_hist_df)
print("onehotencoding: ")
display(teVoorspellenData.head())
# Target encoding
targetOneHotEnc = target(teVoorspellenData)
print("targetencoding: ")
display(targetOneHotEnc.head())

onehotencoding: 


,FullDate,date,Programma,Lengte_sec,Kijkers,Temperatuur,Gevoelstemp,Regen,Sneeuw,Weercode,Bewolking,Windsnelheid,Zonnenschijn,isWeekend,Kijkers_lag_1,Kijkers_lag_2,Kijkers_lag_3,hour_0,hour_1,hour_2,hour_6,hour_7,hour_8,hour_9,hour_10,hour_11,hour_12,hour_13,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23,Kanaal_AB3,Kanaal_CANVAS,Kanaal_CAZ,Kanaal_Canvas,Kanaal_DAZN_PRO_LEAGUE_1_(NL),Kanaal_EEN,Kanaal_ELEVEN_PRO_LEAGUE_1_NL,Kanaal_EUROSPORT_1_(NL),Kanaal_KETNET,Kanaal_LA_UNE,Kanaal_OP_12,Kanaal_PLAY4,Kanaal_PLAY5,Kanaal_PLAY6,Kanaal_PLAY_SPORTS_OPEN,Kanaal_Q2,Kanaal_RTL-TVI,Kanaal_TF1,Kanaal_VIER,Kanaal_VIJF,Kanaal_VITAYA,Kanaal_VRT_1,Kanaal_VRT_CANVAS,Kanaal_VTM,Kanaal_VTM2,Kanaal_VTM3,Kanaal_VTM4,Kanaal_VTM_GOLD,Kanaal_ZES,isFeestdag_0,isFeestdag_1,Weekdag_0,Weekdag_1,Weekdag_2,Weekdag_3,Weekdag_4,Weekdag_5,Weekdag_6,Seizoen_herfst,Seizoen_lente,Seizoen_winter,Seizoen_zomer
193,2025-04-24 21:34:53,2025-04-24,112 HULP IS ONDERWEG,2394,267015,10.6,8.4,0.0,0.0,3,100,13.0,0.0,0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
331,2025-05-01 21:20:48,2025-05-01,112 HULP IS ONDERWEG,2704,284605,24.9,23.3,0.0,0.0,3,100,6.9,0.0,0,267015,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
72,2025-04-18 22:11:57,2025-04-18,9-1-1,2467,255485,12.0,9.0,0.0,0.0,3,100,10.8,0.0,0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
210,2025-04-25 22:28:10,2025-04-25,9-1-1,2437,275338,10.7,8.9,0.0,0.0,0,3,9.6,0.0,0,255485,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
352,2025-05-02 22:40:33,2025-05-02,9-1-1,2427,209967,17.0,15.4,0.0,0.0,1,27,11.7,0.0,0,275338,255485,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


targetencoding: 


,FullDate,Kijkers,Sneeuw,Weercode,isWeekend,Kijkers_lag_1,Kijkers_lag_2,Kijkers_lag_3,hour_0,hour_1,hour_2,hour_6,hour_7,hour_8,hour_9,hour_10,hour_11,hour_12,hour_13,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23,Kanaal_AB3,Kanaal_CANVAS,Kanaal_CAZ,Kanaal_Canvas,Kanaal_DAZN_PRO_LEAGUE_1_(NL),Kanaal_EEN,Kanaal_ELEVEN_PRO_LEAGUE_1_NL,Kanaal_EUROSPORT_1_(NL),Kanaal_KETNET,Kanaal_LA_UNE,Kanaal_OP_12,Kanaal_PLAY4,Kanaal_PLAY5,Kanaal_PLAY6,Kanaal_PLAY_SPORTS_OPEN,Kanaal_Q2,Kanaal_RTL-TVI,Kanaal_TF1,Kanaal_VIER,Kanaal_VIJF,Kanaal_VITAYA,Kanaal_VRT_1,Kanaal_VRT_CANVAS,Kanaal_VTM,Kanaal_VTM2,Kanaal_VTM3,Kanaal_VTM4,Kanaal_VTM_GOLD,Kanaal_ZES,isFeestdag_0,isFeestdag_1,Weekdag_0,Weekdag_1,Weekdag_2,Weekdag_3,Weekdag_4,Weekdag_5,Weekdag_6,Seizoen_herfst,Seizoen_lente,Seizoen_winter,Seizoen_zomer,date,Programma,Lengte_sec,Temperatuur,Gevoelstemp,Regen,Bewolking,Windsnelheid,Zonnenschijn
193,2025-04-24 21:34:53,267015,0.0,3,0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2206.432619,1664.888486,2394,10.6,8.4,0.0,100,13.0,0.0
331,2025-05-01 21:20:48,284605,0.0,3,0,267015,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2206.432619,1664.888486,2704,24.9,23.3,0.0,100,6.9,0.0
72,2025-04-18 22:11:57,255485,0.0,3,0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2206.432619,107.094066,2467,12.0,9.0,0.0,100,10.8,0.0
210,2025-04-25 22:28:10,275338,0.0,0,0,255485,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2206.432619,107.094066,2437,10.7,8.9,0.0,3,9.6,0.0
352,2025-05-02 22:40:33,209967,0.0,1,0,275338,255485,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2206.432619,107.094066,2427,17.0,15.4,0.0,27,11.7,0.0


In [37]:
teVoorspellenData = targetOneHotEnc.select_dtypes(include=[np.number])
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
teVoorspellenData.dtypes

Sneeuw                           float64
Weercode                           int64
isWeekend                          int64
hour_0                           float64
hour_1                           float64
hour_2                           float64
hour_6                           float64
hour_7                           float64
hour_8                           float64
hour_9                           float64
hour_10                          float64
hour_11                          float64
hour_12                          float64
hour_13                          float64
hour_14                          float64
hour_15                          float64
hour_16                          float64
hour_17                          float64
hour_18                          float64
hour_19                          float64
hour_20                          float64
hour_21                          float64
hour_22                          float64
hour_23                          float64
Kanaal_AB3      

In [38]:
try:
    with open('./models/optunaBestModel.pkl', 'rb') as file:
        lgbm = pickle.load(file)
except Exception as e:
    print("Kon model niet laden:", e)
    exit(1)

predictions = lgbm.predict(teVoorspellenData)
display(predictions)

LightGBMError: The number of features in data (75) is not the same as it was in training data (78).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.